In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [3]:
import numpy as np
from embedder import Embedder

embedder = Embedder()
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(v[0])

-0.02058200593003704


In [4]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_content = None

for doc in documents:
    if doc['filename'] == target_file:
        doc_content = doc['content']
        break
        
v_doc = embedder.encode(doc_content)
similarity = np.dot(v, v_doc)
print(similarity)

0.3610702814461231


In [8]:
import numpy as np
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_texts = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(chunk_texts)

scores = X.dot(v)

highest_scoring_idx = np.argmax(scores)
best_chunk = chunks[highest_scoring_idx]

best_chunk

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [14]:
import numpy as np
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])

chunk_texts = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(chunk_texts) # Shape: (num_chunks, 384)

vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query) 
results = vindex.search(query_vector, num_results=5)


print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [33]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"
query_vector = embedder.encode(query)

text_results = text_index.search(query=query, num_results=5)
vector_results = vindex.search(query_vector, num_results=5)

text_files = {doc['filename'] for doc in text_results}
vector_files = {doc['filename'] for doc in vector_results}

print("In Vector results but NOT in Text results:")
print(vector_files - text_files)

In Vector results but NOT in Text results:
{'02-vector-search/lessons/08-pgvector.md'}


In [20]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [34]:
que = "How do I give the model access to tools?"

q_vector = embedder.encode(que)
texts_results = text_index.search(que, num_results=5)
vector_results = vindex.search(q_vector, num_results=5)

In [36]:
results = rrf([vector_results, texts_results])
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'